In [2]:
# Install the missing library first
!pip install ptflops

import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, 
                             roc_auc_score, confusion_matrix, roc_curve, auc)
from sklearn.preprocessing import label_binarize
from ptflops import get_model_complexity_info

# 1. CONFIGURATION

In [3]:
MODEL_NAME = "MobileNetV2"
TEST_SIZE = 0.10  # Change to 0.20, 0.30, etc. for your 10 separate notebooks
RATIO_LABEL = "90:10"
INPUT_SIZE = (224, 224) 
BATCH_SIZE = 64
EPOCHS = 50

# 2. GPU SETUP

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {device}")

Using Device: cuda


# 3. DATA & PATHS

In [7]:
CSV_FILE = "/kaggle/input/datasets/deadcardassian/pm25vision/train/metadata.csv"
IMG_DIR = "/kaggle/input/datasets/deadcardassian/pm25vision/train/images"

df = pd.read_csv(CSV_FILE)
df.iloc[:, 1] = (df.iloc[:, 1] - df.iloc[:, 1].min()).clip(0, 5)

class PM25Dataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        img_name = os.path.basename(str(self.df.iloc[idx, 0]))
        if not img_name.lower().endswith('.jpg'): img_name += '.jpg'
        img_path = os.path.join(IMG_DIR, img_name)
        image = Image.open(img_path).convert('RGB')
        if self.transform: image = self.transform(image)
        return image, int(self.df.iloc[idx, 1])

transform = transforms.Compose([
    transforms.Resize(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 4. MODEL & COMPLEXITY

In [9]:
model = models.mobilenet_v2(weights='IMAGENET1K_V1')
model.classifier[1] = nn.Linear(model.last_channel, 6)
model = model.to(device)

# GFLOPs Calculation
macs, params = get_model_complexity_info(model, (3, 224, 224), as_strings=True, print_per_layer_stat=False)
print(f"Complexity: {macs} MACs, Params: {params}")

Complexity: 319.03 MMac MACs, Params: 2.23 M


# 5. TRAINING

In [10]:
train_df, test_df = train_test_split(df, test_size=TEST_SIZE, stratify=df.iloc[:,1], random_state=42)
train_loader = DataLoader(PM25Dataset(train_df, transform), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(PM25Dataset(test_df, transform), batch_size=BATCH_SIZE, num_workers=2)

optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

start_train_wall = time.time()
for epoch in range(EPOCHS):
    model.train()
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), lbls)
        loss.backward()
        optimizer.step()

train_wall_time = time.time() - start_train_wall